<span style="font-size:1.5em;">
This notebook presumes you understand the basics of working with DataCubes covered in the <a href=introduction-to-lk-datacubes.ipynb>introduction-to-lk-datacubes.ipynb</a> notebook
</span>

In [ ]:
from lightkurve import DataCube, ErrorCube
import numpy as np

%load_ext autoreload
%autoreload 2

In [ ]:
from lksearch import TESSSearch
from astropy.coordinates import SkyCoord
from astropy.io import fits

search_input = (84.291190, -80.469170)
search_input = SkyCoord(*search_input, unit="deg", frame="icrs")
search_result = TESSSearch(search_input, hlsp=False)

# You can filter two different ways to get the same result.
# search_result.filter_table(pipeline='TESScut')
downloads_table = search_result.tesscut.filter_table(sector=1).download(TESScut_size=10)
hdulist = fits.open(downloads_table["Local Path"][0])

# Prepare DataCube and ErrorCube

In [ ]:
flux_array = hdulist[1].data["FLUX"].astype(float)
flux_err_array = hdulist[1].data["FLUX_ERR"].astype(float)
time = hdulist[1].data["TIME"].astype(float)
# Spacecraft time correction for relativistic effects
time_corr = hdulist[1].data["TIMECORR"].astype(float)
# Reference for the 0-index row and column positions on the CCD
c0, r0 = hdulist[1].header["1CRV4P"], hdulist[1].header["2CRV4P"]
row, col = np.arange(flux_array.shape[1]) + r0, np.arange(flux_array.shape[2]) + c0

In [ ]:
flux = DataCube(
    flux_array,
    time_indices={"btjd": time, "spacecraft_time": time - time_corr},
    row_indices={"pixel_row": row},
    col_indices={"pixel_column": col},
)

In [ ]:
flux_err = ErrorCube(
    flux_err_array,
    time_indices={"btjd": time, "spacecraft_time": time - time_corr},
    row_indices={"pixel_row": row},
    col_indices={"pixel_column": col},
)

## Imposing an aperture mask where `mean`(flux of pixel) > 10,000

In [ ]:
aper = flux_array.mean(axis=0) > 10000
bkg_aper = flux_array.mean(axis=0) < 4000

time_mask = hdulist[1].data["QUALITY"] == 0

Imposing an aperture on a DataCube returns a DataFrame providing a time series for each pixel.

In [ ]:
flux[:, aper]

## Sum of flux in aperture, per cadence

Summing the flux over the entire aperture produces a single quantity per cadence, the result is a `DataSeries`. 

In [ ]:
flux[:, aper].sum(axis=1)

## Sum of flux error in aperture, per cadence. 
Flux error is added via [root-mean-square.](https://en.wikipedia.org/wiki/Propagation_of_uncertainty)

In [ ]:
flux_err[:, aper].sum(axis=1)

# Plot the light curve

In [ ]:
import matplotlib.pyplot as plt

time = np.asarray(flux.btjd)
bkg = flux[:, bkg_aper].mean(axis=1)
bkg_err = flux_err[:, bkg_aper].mean(axis=1)

bkg -= bkg.median()

lc = flux[:, aper].sum(axis=1)
lc_err = flux_err[:, aper].sum(axis=1)

plt.figure()
plt.title("Raw")
plt.errorbar(time, lc.values, lc_err.values, ls="", marker=".", c="k")
# plt.ylim(.88e6, .94e6)

lc = flux[:, aper].sum(axis=1) - (bkg * aper.sum())
lc_err = flux_err[:, aper].sum(axis=1) + (bkg_err * aper.sum())

plt.figure()
plt.title("Background Subtracted")
plt.errorbar(time, lc.values, lc_err.values, ls="", marker=".", c="k")
# plt.ylim(0.88e6, 0.94e6)

# Downsampling

## Temporal

In [ ]:
flux.downsample(5)

In [ ]:
time = np.asarray(flux.downsample(5).btjd)
lc = flux.downsample(5).sum(axis=1)
lc_err = flux_err.downsample(5).sum(axis=1)

plt.figure()
plt.title("Raw")
plt.errorbar(time, lc.values, lc_err.values, ls="", marker=".", c="k")

## Spatial

In [ ]:
flux.spatial_downsample(2)

In [ ]:
time = np.asarray(flux.spatial_downsample(2).btjd)
lc = flux.spatial_downsample(2).sum(axis=1)
lc_err = flux_err.spatial_downsample(2).sum(axis=1)

plt.figure()
plt.title("Raw")
plt.errorbar(time, lc.values, lc_err.values, ls="", marker=".", c="k")

# Next Up
* [Core functionality](datacube-operations.ipynb)